In [1]:
# Import the libraries we need for modelling
import pandas as pd        # pandas: for handling our data table
import sqlite3             # sqlite3: to read our saved features from the database
from sklearn.model_selection import train_test_split  # the tool that splits data into train/test piles

# Open the database that holds our engineered features from Phase 2
conn = sqlite3.connect(r'C:\Users\User\Desktop\football-xg-model\data\football_xg.db')

# Load the 'shot_features' table (the model-ready data we built in Phase 2)
# "SELECT * FROM shot_features" = grab every row and column from that table
df = pd.read_sql("SELECT * FROM shot_features", conn)

# Confirm it loaded (should be 9168 rows)
print("Loaded", df.shape[0], "shots with features")
df.head()

Loaded 9168 shots with features


,id,match_id,player,team,distance,angle,is_header,is_penalty,is_open_play,shot_statsbomb_xg,goal
0,a1501eed-1c34-4060-a92a-6bb8cef36c39,3825739,Jonathan Rodríguez Menéndez,Sporting Gijón,19.994499,0.353466,0,0,1,0.125582,0
1,085c74bd-911f-4e55-965d-4c7c94088a67,3825739,Gareth Frank Bale,Real Madrid,5.521775,1.055740,1,0,1,0.245234,1
2,10c2092d-0869-4f06-b6d7-e4c2bb99a07b,3825739,Cristiano Ronaldo dos Santos Aveiro,Real Madrid,18.160396,0.396012,0,0,1,0.060526,1
3,8f9dac21-9fe8-4f59-be34-87cd48b1c057,3825739,Luka Modrić,Real Madrid,29.463537,0.268445,0,0,1,0.024467,0
4,4ad739ee-1133-491f-9760-b19bb6d407e2,3825739,Karim Benzema,Real Madrid,11.016805,0.691321,0,0,1,0.088242,1


In [2]:
# ---- SEPARATE THE FEATURES (inputs) FROM THE TARGET (answer) ----

# X = the features: the clues the model uses to make its guess.
# We pick our 5 engineered columns. This is what the model "sees".
X = df[['distance', 'angle', 'is_header', 'is_penalty', 'is_open_play']]

# y = the target: the actual answer we want it to predict (1 = goal, 0 = no goal).
y = df['goal']

# ---- SPLIT INTO TRAINING AND TESTING PILES ----

# train_test_split randomly divides our shots into two groups:
#   - X_train, y_train: the ~80% the model LEARNS from
#   - X_test,  y_test:  the ~20% we HIDE, to test the model on unseen shots
# test_size=0.2 means 20% goes to testing.
# random_state=42 just makes the random split the same every time we run it (so results are reproducible).
# stratify=y makes sure both piles keep the same 11% goal rate (important with imbalanced data!).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Print the sizes of each pile so we can see the split worked
print("Training shots:", X_train.shape[0])   # should be ~7334
print("Testing shots: ", X_test.shape[0])    # should be ~1834

Training shots: 7334
Testing shots:  1834


In [3]:
# Import the logistic regression model from scikit-learn
# Logistic regression is a simple, classic model that predicts a PROBABILITY (0 to 1) —
# perfect for xG, since we want "chance of goal", not just yes/no.
from sklearn.linear_model import LogisticRegression

# Create the model. max_iter=1000 just gives it enough attempts to find the best fit
# (the default sometimes stops too early and warns; 1000 is plenty).
baseline_model = LogisticRegression(max_iter=1000)

# TRAIN the model: .fit() shows it the training shots (X_train = the clues)
# and the correct answers (y_train = did it score), so it learns the pattern.
baseline_model.fit(X_train, y_train)

# Now the model is trained! Let's make it predict on the HIDDEN test shots.
# .predict_proba() gives the probability of each outcome.
# [:, 1] grabs the probability of class "1" (goal) — that's our xG value for each shot.
xg_predictions = baseline_model.predict_proba(X_test)[:, 1]

# Show the first 10 predicted xG values so we can see them
# .round(3) makes them readable (3 decimal places)
print("First 10 predicted xG values:")
print(xg_predictions[:10].round(3))

First 10 predicted xG values:
[0.041 0.075 0.031 0.534 0.515 0.153 0.006 0.03  0.087 0.189]


In [4]:
# Import the proper evaluation metrics for probability predictions
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

# ---- METRIC 1: LOG LOSS ----
# Log loss punishes confident-but-wrong predictions heavily.
# LOWER is better. It's the right metric for probabilities (not accuracy).
ll = log_loss(y_test, xg_predictions)

# ---- METRIC 2: BRIER SCORE ----
# Brier score = average squared difference between predicted xG and actual outcome.
# Also LOWER is better. Another honest metric for probabilities.
brier = brier_score_loss(y_test, xg_predictions)

# ---- METRIC 3: ROC AUC ----
# AUC measures how well the model RANKS shots (good chances above bad ones).
# 0.5 = random guessing, 1.0 = perfect. HIGHER is better.
auc = roc_auc_score(y_test, xg_predictions)

# Print all three so we can judge the model
print("Log loss:  ", round(ll, 4), "(lower is better)")
print("Brier score:", round(brier, 4), "(lower is better)")
print("ROC AUC:   ", round(auc, 4), "(higher is better, 0.5 = random)")

Log loss:   0.2744 (lower is better)
Brier score: 0.0788 (lower is better)
ROC AUC:    0.8097 (higher is better, 0.5 = random)


In [5]:
# Get StatsBomb's own xG values for the SAME test shots we tested our model on.
# X_test.index gives the row positions of our test shots, so we grab StatsBomb's xG for exactly those.
statsbomb_xg_test = df.loc[X_test.index, 'shot_statsbomb_xg']

# Now score StatsBomb's model using the same three metrics, so it's a fair head-to-head.
sb_ll = log_loss(y_test, statsbomb_xg_test)             # their log loss
sb_brier = brier_score_loss(y_test, statsbomb_xg_test)  # their brier score
sb_auc = roc_auc_score(y_test, statsbomb_xg_test)       # their AUC

# Print a side-by-side comparison: YOUR model vs STATSBOMB's professional model
print("            YOUR model  |  StatsBomb")
print(f"Log loss:    {round(ll,4)}      |  {round(sb_ll,4)}")
print(f"Brier score: {round(brier,4)}     |  {round(sb_brier,4)}")
print(f"ROC AUC:     {round(auc,4)}     |  {round(sb_auc,4)}")

            YOUR model  |  StatsBomb
Log loss:    0.2744      |  0.2468
Brier score: 0.0788     |  0.0703
ROC AUC:     0.8097     |  0.8506


In [6]:
# Import XGBoost — a powerful "gradient boosting" model that's very popular in industry.
# It builds many small decision trees, each correcting the previous one's mistakes.
from xgboost import XGBClassifier

# Create the XGBoost model with some sensible settings:
#   n_estimators=200  -> build 200 little trees
#   max_depth=4       -> keep each tree shallow (avoids "memorising" the training data)
#   learning_rate=0.05-> each tree makes small careful improvements (steadier, less overfitting)
#   eval_metric='logloss' -> tells XGBoost to optimise for good probabilities
#   random_state=42   -> makes results reproducible (same every run)
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    eval_metric='logloss',
    random_state=42
)

# TRAIN it on the same training shots our baseline used (fair comparison)
xgb_model.fit(X_train, y_train)

# Predict xG on the SAME hidden test shots
# [:, 1] again grabs the probability of "goal" for each shot
xgb_predictions = xgb_model.predict_proba(X_test)[:, 1]

# Score it with the same three metrics
xgb_ll = log_loss(y_test, xgb_predictions)
xgb_brier = brier_score_loss(y_test, xgb_predictions)
xgb_auc = roc_auc_score(y_test, xgb_predictions)

# Print the FULL three-way comparison: baseline vs XGBoost vs StatsBomb
print("             Baseline  |  XGBoost  |  StatsBomb")
print(f"Log loss:    {round(ll,4)}    |  {round(xgb_ll,4)}   |  {round(sb_ll,4)}")
print(f"Brier score: {round(brier,4)}   |  {round(xgb_brier,4)}  |  {round(sb_brier,4)}")
print(f"ROC AUC:     {round(auc,4)}   |  {round(xgb_auc,4)}  |  {round(sb_auc,4)}")

             Baseline  |  XGBoost  |  StatsBomb
Log loss:    0.2744    |  0.2769   |  0.2468
Brier score: 0.0788   |  0.0808  |  0.0703
ROC AUC:     0.8097   |  0.8133  |  0.8506
